In [123]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import holoviews as hv
import hvplot.pandas
import panel as pn
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as mp

from scipy import stats
from pathlib import Path
from pprint import pprint
from holoviews import opts
from bokeh.io import output_notebook


output_notebook()
hv.extension('bokeh')

font_dict = {'title': 16, 'labels': 14, 'ticks': 12, 'legend': 12}
hv.opts.defaults(
    hv.opts.Curve(width=600, height=400, tools=['hover'], fontsize=font_dict),
    hv.opts.Scatter(width=600, height=400, size=8, tools=['hover'], fontsize=font_dict),
    hv.opts.Histogram(width=600, height=400, fontsize=font_dict),
    hv.opts.Bars(width=600, height=400, fontsize=font_dict),
)

Loading BokehJS ...

In [142]:
monkey = 'fiona' # 'yasmin'  or 'fiona' 
base_path = Path.cwd().parent / 'data' / 'csst_trials_pkls'
filepath = base_path / f'all_{monkey}_CSST_trials_df.pkl'
df = pd.read_pickle(filepath)

# print(df.info())
# df.iloc[:2]
print(df.columns)
df

Index(['blinks', 'dir', 'direction', 'filename', 'first_relevant_saccade',
       'flags', 'go_cue', 'hPos', 'hVel', 'neural_data', 'reaction_time',
       'saccades', 'screen_rotation', 'segs_durations', 'segs_times', 'set',
       'speed', 'ssd_len', 'ssd_number', 'stop_cue', 'trial_failed',
       'trial_length', 'trial_name', 'trial_number', 'trial_session', 'type',
       'vPos', 'vVel'],
      dtype='object')


,blinks,dir,direction,filename,first_relevant_saccade,flags,go_cue,hPos,hVel,neural_data,...,ssd_number,stop_cue,trial_failed,trial_length,trial_name,trial_number,trial_session,type,vPos,vVel
0,None,180,L,fi210824a.0614,"[1382, 1457]",8206,1054,"[11.275, 11.275, 11.275, 11.275, 11.275, 11.27...","[0.0, 0.0, 0.4594490287247533, 1.3783470861742...","{0: [884.4], 1: [154.42, 329.18, 1478.9], 3: [...",...,2.0,1186.0,False,2205,CONT_L_SSD2,0614,fi210824a,CONT,"[-0.05, -0.05, -0.05, -0.05, -0.05, -0.05, 0.0...","[-3.1242533953283225, -3.1242533953283225, -4...."
1,None,180,L,fi210824a.0520,"[1100, 1172]",8206,914,"[-11.15, -11.15, -11.15, -11.15, -11.15, -11.1...","[-2.7566941723485194, -2.7566941723485194, -3....","{1: [48.52, 264.55, 585.97, 1032.3], 2: [385.2...",...,NaN,NaN,False,2065,GO_L,0520,fi210824a,GO,"[-1.15, -1.15, -1.15, -1.15, -1.15, -1.175, -1...","[-0.8270082517045559, -0.8270082517045559, 0.1..."
2,None,180,L,fi210824a.1193,"[1099, 1179]",8206,938,"[2.25, 2.25, 2.3, 2.3, 2.3, 2.25, 2.225, 2.225...","[-188.19032216565896, -188.19032216565896, -18...","{1: [574.7, 853.02, 1403.57], 3: [8.35, 301.92...",...,3.0,1118.0,False,2089,CONT_L_SSD3,1193,fi210824a,CONT,"[-27.45, -27.45, -27.45, -27.45, -27.45, -27.0...","[0.0, 0.0, 0.0, 0.0, 0.0, 9.280870380240016, 4..."
3,None,180,L,fi210824a.1013,"[1213, 1289]",13326,1081,"[8.425, 8.425, 8.375, 8.375, 8.375, 8.4, 8.4, ...","[1.286457280429309, 1.286457280429309, -0.9188...","{2: [1954.73], 5: [434.05, 484.13, 547.7, 851....",...,3.0,1261.0,True,1961,STOP_L_SSD3,1013,fi210824a,STOP,"[0.275, 0.275, 0.275, 0.275, 0.275, 0.275, 0.2...","[0.9188980574495066, 0.9188980574495066, 0.0, ..."
4,None,0,R,fi210824a.1257,NaN,8194,1024,"[-11.475, -11.475, -11.475, -11.475, -11.475, ...","[3.767482035542977, 3.767482035542977, 3.85937...","{0: [923.35], 1: [1125], 5: [1026.4, 1359.92, ...",...,2.0,1156.0,True,1476,CONT_R_SSD2,1257,fi210824a,CONT,"[-1.825, -1.825, -1.8, -1.8, -1.825, -1.825, -...","[0.27566941723485194, 0.27566941723485194, 1.0..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110353,None,0,R,fi211020a.0230,"[1213, 1285]",8206,943,"[-9.925, -9.925, -9.925, -9.925, -9.95, -9.95,...","[-0.09188980574495066, -0.09188980574495066, -...","{0: [1937.51], 1: [24.51, 42.58, 111.86, 173.7...",...,NaN,NaN,False,2094,GO_R,0230,fi211020a,GO,"[-1.35, -1.35, -1.45, -1.45, -1.35, -1.35, -1....","[-1.8377961148990132, -1.8377961148990132, -3...."
110354,None,0,R,fi211020a.0832,"[1397, 1470]",8206,1070,"[-11.925, -11.925, -11.85, -11.85, -11.875, -1...","[-4.686380092992484, -4.686380092992484, 0.0, ...","{0: [1027.96], 1: [19.86, 91.81, 177.64, 261.5...",...,NaN,NaN,False,2221,GO_R,0832,fi211020a,GO,"[-0.7, -0.7, -0.625, -0.625, -0.675, -0.675, -...","[-3.3080330068182238, -3.3080330068182238, 0.2..."
110355,None,180,L,fi211020a.1304,"[1210, 1287]",8206,1051,"[-11.95, -11.95, -11.95, -11.95, -11.95, -11.9...","[2.389134949368717, 0.5513388344697039, 0.5513...","{0: [16.69, 27.66, 37.08, 89.74, 120.56, 218.7...",...,NaN,NaN,False,2202,GO_L,1304,fi211020a,GO,"[0.725, 0.75, 0.75, 0.775, 0.75, 0.75, 0.725, ...","[-1.6540165034091119, -1.1026776689394078, -1...."
110356,None,180,L,fi211020a.0719,"[1123, 1197]",8194,1001,"[0.0, 0.0, -0.1, -0.175, -0.175, -0.175, -0.37...","[-51.090731994192566, -51.090731994192566, -50...","{0: [521.64, 535.58, 677.96, 741.66, 963.88, 9...",...,NaN,NaN,True,2057,GO_L,0719,fi211020a,GO,"[-4.425, -4.425, -3.35, -3.0, -2.65, -2.65, -1...","[188.098432359914, 188.098432359914, 188.09843..."


In [140]:
def get_cells_count_in_neural_data(trial):
    neural_data = trial['neural_data']
    if neural_data is None:
        return 0
    return len(neural_data.keys())  # number of cells

trial = df.iloc[0]
num_cells = get_cells_count_in_neural_data(trial)
print(f'Number of cells in first trial: {num_cells}')
trial['neural_data']

df.apply(get_cells_count_in_neural_data, axis=1)

Number of cells in first trial: 9


0          9
1          9
2          9
3         10
4         10
          ..
110353    72
110354    73
110355    54
110356    73
110357    71
Length: 110358, dtype: int64

In [126]:
df['filename'].apply(lambda x: x.split('.')[0][-1]).unique()

array(['a'], dtype=object)

In [127]:
# Drop unnecessary columns
cols_to_drop = [
    'vPos', 'hPos', 'vVel', 'hVel', 'speed',
    'set', 'direction'
]
df.drop(columns=cols_to_drop, inplace=True)
print(f"DataFrame shape after dropping columns: {df.shape}")
# df.head()

DataFrame shape after dropping columns: (110358, 21)


In [128]:
# reorder columns
new_order = [
    'filename', 'trial_name', 'reaction_time', 
    'go_cue', 'stop_cue', 'trial_failed', 
    'first_relevant_saccade', 'segs_durations', 'segs_times',
    'trial_length', 'ssd_len', 'ssd_number',
    'screen_rotation', 'neural_data', 'saccades', 
    'blinks', 'dir', 'flags',
    'type', 'trial_session', 'trial_number',
]

df = df[new_order]
# df.head()

In [129]:
df['filename']

0         fi210824a.0614
1         fi210824a.0520
2         fi210824a.1193
3         fi210824a.1013
4         fi210824a.1257
               ...      
110353    fi211020a.0230
110354    fi211020a.0832
110355    fi211020a.1304
110356    fi211020a.0719
110357    fi211020a.0901
Name: filename, Length: 110358, dtype: object

In [130]:
# load monkey's cell db from xlsx file
cell_db_path = Path.cwd().parent / 'data' / f'database_sst'
cell_db = pd.read_excel(cell_db_path / f'SST_{monkey}_cells_db.xlsx')
print(cell_db.info())
cell_db.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5309 entries, 0 to 5308
Data columns (total 24 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   cell_ID              5309 non-null   int64  
 1   session              5309 non-null   object 
 2   cell_type            5309 non-null   object 
 3   electrode            5309 non-null   int64  
 4   template             5309 non-null   int64  
 5   maestro_ID           5309 non-null   int64  
 6   phy_id               0 non-null      float64
 7   phy_channel          0 non-null      float64
 8   file_begin           5309 non-null   int64  
 9   file_end             5309 non-null   int64  
 10  fb_after_stablility  5309 non-null   object 
 11  fe_after_stability   5309 non-null   object 
 12  plexon_session       5309 non-null   object 
 13  grade                5309 non-null   int64  
 14  X                    5309 non-null   int64  
 15  Y                    5309 non-null   i

,cell_ID,session,cell_type,electrode,template,maestro_ID,phy_id,phy_channel,file_begin,file_end,...,X,Y,depth_mm,is_continuous,comments,plex_sorted_file,tmp,sorted,problem,synced_stability
0,9001,fi210628,ctx,1,1,1,NaN,NaN,1,225,...,0,0,4420.0,2,depth micro m is from cortex surface,fi210628a-01.pl2,NaN,1,broken cell two peaks,1
1,9002,fi210629,msn,1,1,1,NaN,NaN,84,163,...,0,-1,10200.0,2,NaN,fi210629a-01.pl2,NaN,1,NaN,1
2,9003,fi210701,msn,1,1,1,NaN,NaN,16,348,...,0,-1,7030.0,2,2 cells multi unit,fi210701a-02.pl2,NaN,1,NaN,1
3,9004,fi210701,tan,1,1,1,NaN,NaN,350,510,...,0,-1,7110.0,2,NaN,fi210701b-02.pl2,NaN,1,NaN,1
4,9005,fi210701,msn,1,2,2,NaN,NaN,16,348,...,0,-1,7030.0,2,NaN,fi210701a-02.pl2,NaN,1,NaN,1


In [131]:
cell_db.columns

Index(['cell_ID', 'session', 'cell_type', 'electrode', 'template',
       'maestro_ID', 'phy_id', 'phy_channel', 'file_begin', 'file_end',
       'fb_after_stablility', 'fe_after_stability', 'plexon_session', 'grade',
       'X', 'Y', 'depth_mm', 'is_continuous', 'comments', 'plex_sorted_file',
       'tmp', 'sorted', 'problem', 'synced_stability'],
      dtype='object')

In [153]:
msn_cells = cell_db[
    cell_db['cell_type'].isin(['msn', 'pu msn']) &
    (cell_db['grade'] <= 100) 
].copy().reset_index(drop=True)
print(f'Number of MSN cells: {len(msn_cells)}')

Number of MSN cells: 4206


In [154]:
tmp = cell_db[
    ~cell_db['fe_after_stability'].apply(
        lambda x: isinstance(x, int)
    )
][['fe_after_stability', 'fb_after_stablility']] 

tmp['fe_after_stability'] = tmp['fe_after_stability'].apply(
    lambda x: np.fromstring(x.strip('[]'), sep=' ')
)

tmp['fb_after_stablility'] = tmp['fb_after_stablility'].apply(
    lambda x: np.fromstring(x.strip('[]'), sep=' ')
)

tmp['comp'] = tmp.apply(
    lambda row: len(row['fe_after_stability']) == len(row['fb_after_stablility']), axis=1
) 

#- cell_db['fb_after_stablility'].astype(int)

In [155]:
def process_cells_stability(row):
    fe = row['fe_after_stability']
    fb = row['fb_after_stablility']

    if isinstance(fe, np.ndarray) and isinstance(fb, np.ndarray):
        return fe, fb
    
    if isinstance(fe, int) and isinstance(fb, int):
        return fe, fb
    
    fe_array = np.fromstring(fe.strip('[]'), sep=' ')
    fb_array = np.fromstring(fb.strip('[]'), sep=' ')
    
    return fe_array, fb_array

msn_cells[['fe_after_stability', 'fb_after_stablility']] = msn_cells.apply(
    process_cells_stability, axis=1, result_type='expand'
)



In [156]:
msn_cells[
    ~msn_cells.apply(
    lambda row: isinstance(row['fe_after_stability'], int), axis=1
)].apply(
    lambda row: type(row['fe_after_stability']), axis=1
).value_counts()

<class 'numpy.ndarray'>    91
Name: count, dtype: int64

In [157]:
(msn_cells['fe_after_stability'] - msn_cells['fb_after_stablility']).apply(
    lambda x: x if not isinstance(x, np.ndarray) else x.sum()
).sum().astype(int)


np.int64(2806628)

In [216]:
minimal_df = msn_cells[
    [
        'session', 'cell_ID', 
        'fe_after_stability', 'fb_after_stablility',
        'maestro_ID', 'plexon_session'    
    ]
]

def get_trials_range(row):
    fe = row['fe_after_stability']
    fb = row['fb_after_stablility']
    
    session = row['session']
    # cell_ID = row['cell_ID']
    # maestro_ID = row['maestro_ID']
    plexon_session = 'a' #row['plexon_session']

    row_trials = []
    if isinstance(fe, int) and isinstance(fb, int):
        for trial_num in range(fb, fe + 1):
            row_trials.append(f"{session}{plexon_session}.{trial_num:04d}")
    elif isinstance(fe, np.ndarray) and isinstance(fb, np.ndarray):
        for start, end in zip(fe, fb):
            for trial_num in range(int(start), int(end) + 1):
                row_trials.append(f"{session}{plexon_session}.{trial_num:04d}")
    else:
        raise ValueError(f"Unexpected types for fe and fb: {type(fe)}, {type(fb)}")
    return row_trials

minimal_df['trials_list'] = minimal_df.apply(get_trials_range, axis=1)


minimal_df['trials_list'].iloc[0]  
    

/tmp/ipykernel_72252/2658607796.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  minimal_df['trials_list'] = minimal_df.apply(get_trials_range, axis=1)


['fi210629a.0084',
 'fi210629a.0085',
 'fi210629a.0086',
 'fi210629a.0087',
 'fi210629a.0088',
 'fi210629a.0089',
 'fi210629a.0090',
 'fi210629a.0091',
 'fi210629a.0092',
 'fi210629a.0093',
 'fi210629a.0094',
 'fi210629a.0095',
 'fi210629a.0096',
 'fi210629a.0097',
 'fi210629a.0098',
 'fi210629a.0099',
 'fi210629a.0100',
 'fi210629a.0101',
 'fi210629a.0102',
 'fi210629a.0103',
 'fi210629a.0104',
 'fi210629a.0105',
 'fi210629a.0106',
 'fi210629a.0107',
 'fi210629a.0108',
 'fi210629a.0109',
 'fi210629a.0110',
 'fi210629a.0111',
 'fi210629a.0112',
 'fi210629a.0113',
 'fi210629a.0114',
 'fi210629a.0115',
 'fi210629a.0116',
 'fi210629a.0117',
 'fi210629a.0118',
 'fi210629a.0119',
 'fi210629a.0120',
 'fi210629a.0121',
 'fi210629a.0122',
 'fi210629a.0123',
 'fi210629a.0124',
 'fi210629a.0125',
 'fi210629a.0126',
 'fi210629a.0127',
 'fi210629a.0128',
 'fi210629a.0129',
 'fi210629a.0130',
 'fi210629a.0131',
 'fi210629a.0132',
 'fi210629a.0133',
 'fi210629a.0134',
 'fi210629a.0135',
 'fi210629a.

In [217]:
trials_set = set()

for i, row in minimal_df.iterrows():
    trials = row['trials_list']
    trials_set.update(trials)

len(trials_set)
trials_set

{'fi210726a.0709',
 'fi211108a.0675',
 'fi211018a.1196',
 'fi211104a.2339',
 'fi211123a.2072',
 'fi210902a.1033',
 'fi210729a.0297',
 'fi210927a.0316',
 'fi210719a.1463',
 'fi210819a.1170',
 'fi211018a.0910',
 'fi210818a.0428',
 'fi210823a.0810',
 'fi211026a.0448',
 'fi211117a.1780',
 'fi210714a.1044',
 'fi211118a.2056',
 'fi211012a.1606',
 'fi210819a.0202',
 'fi210726a.0366',
 'fi211115a.1343',
 'fi210816a.0356',
 'fi211014a.0592',
 'fi211026a.0984',
 'fi210908a.0769',
 'fi210831a.0568',
 'fi210908a.0002',
 'fi210815a.0523',
 'fi210812a.1628',
 'fi211020a.0257',
 'fi210729a.0683',
 'fi210907a.0237',
 'fi210908a.0384',
 'fi211109a.0582',
 'fi211104a.1101',
 'fi211019a.0546',
 'fi211118a.2043',
 'fi210811a.1482',
 'fi211017a.0184',
 'fi211018a.1222',
 'fi211118a.2045',
 'fi210729a.0532',
 'fi211104a.1059',
 'fi210819a.1087',
 'fi211026a.0107',
 'fi210811a.0113',
 'fi210823a.1178',
 'fi210815a.0830',
 'fi211125a.1289',
 'fi210817a.0904',
 'fi210819a.0963',
 'fi211109a.0246',
 'fi211018a.

In [218]:
len(set(df['filename'].to_list()).intersection(trials_set))

82558

In [219]:
base_path

PosixPath('/home/barak/Projects/population-analysis/data/csst_trials_pkls')

In [220]:
fi_maestro_base_path = base_path.parent / 'from_indra' / 'fiona_sst'

# Get set of all filenames in subfolders
all_files_on_disk = set(p.name for p in fi_maestro_base_path.rglob('*') if p.is_file())

# Check existence
found_trials = [t for t in trials_set if t in all_files_on_disk]
missing_trials = [t for t in trials_set if t not in all_files_on_disk]

print(f"Total trials in set: {len(trials_set)}")
print(f"Found on disk: {len(found_trials)}")
print(f"Missing: {len(missing_trials)}")

if missing_trials:
    print(f"Example missing: {missing_trials[:5]}")



Total trials in set: 93663
Found on disk: 93643
Missing: 20
Example missing: ['fi211125a.1780', 'fi211125a.1768', 'fi211125a.1769', 'fi211125a.1771', 'fi211125a.1772']


In [221]:
print([t for t in missing_trials if 'a' in t])

['fi211125a.1780', 'fi211125a.1768', 'fi211125a.1769', 'fi211125a.1771', 'fi211125a.1772', 'fi211125a.1773', 'fi211125a.1776', 'fi211125a.1777', 'fi211125a.1762', 'fi211125a.1775', 'fi211125a.1761', 'fi211125a.1765', 'fi211125a.1763', 'fi211125a.1766', 'fi211125a.1767', 'fi211125a.1770', 'fi211125a.1764', 'fi211125a.1779', 'fi211125a.1778', 'fi211125a.1774']


In [222]:
msn_cells.apply(
    lambda row: (f"{row['session']}{row['plexon_session']}", row['cell_type']), 
    axis=1 #, result_type='expand'
).nunique()

162

In [223]:
# Get unique sessions from the DataFrame
unique_msn_sessions = msn_cells['session'].unique()

# Check existence of session folders
missing_session_dirs = []
for session in unique_msn_sessions:
    session_dir = fi_maestro_base_path / session
    if not session_dir.exists() or not session_dir.is_dir():
        missing_session_dirs.append(session)

print(f"Total unique MSN sessions: {len(unique_msn_sessions)}")
print(f"Missing session directories: {len(missing_session_dirs)}")

if missing_session_dirs:
    print(f"Missing directories: {missing_session_dirs}")
else:
    print("All session directories exist.")

Total unique MSN sessions: 66
Missing session directories: 0
All session directories exist.


In [224]:
msn_cells

,cell_ID,session,cell_type,electrode,template,maestro_ID,phy_id,phy_channel,file_begin,file_end,...,X,Y,depth_mm,is_continuous,comments,plex_sorted_file,tmp,sorted,problem,synced_stability
0,9002,fi210629,msn,1,1,1,NaN,NaN,84,163,...,0,-1,10200.0,2,NaN,fi210629a-01.pl2,NaN,1,NaN,1
1,9003,fi210701,msn,1,1,1,NaN,NaN,16,348,...,0,-1,7030.0,2,2 cells multi unit,fi210701a-02.pl2,NaN,1,NaN,1
2,9005,fi210701,msn,1,2,2,NaN,NaN,16,348,...,0,-1,7030.0,2,NaN,fi210701a-02.pl2,NaN,1,NaN,1
3,9006,fi210704,msn,1,1,1,NaN,NaN,15,191,...,0,-3,5120.0,2,NaN,fi210704a-02.pl2,NaN,1,NaN,1
4,9007,fi210705,msn,1,1,1,NaN,NaN,35,317,...,0,-5,4670.0,2,NaN,fi210705a-01.pl2,NaN,1,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4201,4376,fi211125,pu msn,97,1,28,NaN,NaN,1077,1780,...,-8,-1,NaN,2,NaN,fi211125b-06.pl2,17650.0,1,NaN,1
4202,4377,fi211125,pu msn,102,1,29,NaN,NaN,1077,1780,...,-8,-1,NaN,2,NaN,fi211125b-06.pl2,17650.0,1,NaN,1
4203,4379,fi211125,pu msn,119,1,31,NaN,NaN,1077,1780,...,-8,-1,NaN,2,NaN,fi211125b-06.pl2,17650.0,1,NaN,1
4204,4380,fi211125,pu msn,120,1,32,NaN,NaN,1077,1780,...,-8,-1,NaN,2,NaN,fi211125b-06.pl2,17650.0,1,NaN,1


In [225]:
cell_db['session'].unique().shape
cell_db[cell_db['cell_type'] == 'msn']

,cell_ID,session,cell_type,electrode,template,maestro_ID,phy_id,phy_channel,file_begin,file_end,...,X,Y,depth_mm,is_continuous,comments,plex_sorted_file,tmp,sorted,problem,synced_stability
1,9002,fi210629,msn,1,1,1,NaN,NaN,84,163,...,0,-1,10200.0,2,NaN,fi210629a-01.pl2,NaN,1,NaN,1
2,9003,fi210701,msn,1,1,1,NaN,NaN,16,348,...,0,-1,7030.0,2,2 cells multi unit,fi210701a-02.pl2,NaN,1,NaN,1
4,9005,fi210701,msn,1,2,2,NaN,NaN,16,348,...,0,-1,7030.0,2,NaN,fi210701a-02.pl2,NaN,1,NaN,1
5,9006,fi210704,msn,1,1,1,NaN,NaN,15,191,...,0,-3,5120.0,2,NaN,fi210704a-02.pl2,NaN,1,NaN,1
6,9007,fi210705,msn,1,1,1,NaN,NaN,35,317,...,0,-5,4670.0,2,NaN,fi210705a-01.pl2,NaN,1,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4951,4024,fi211118,msn,19,1,33,NaN,NaN,1063,2433,...,0,-4,NaN,2,NaN,fi211118b-06.pl2,12800.0,1,NaN,1
4952,4025,fi211118,msn,20,1,34,NaN,NaN,1063,2433,...,0,-4,NaN,2,NaN,fi211118b-06.pl2,12800.0,1,NaN,1
4953,4026,fi211118,msn,21,1,35,NaN,NaN,1063,2433,...,0,-4,NaN,2,NaN,fi211118b-06.pl2,12800.0,1,NaN,1
4954,4027,fi211118,msn,21,2,36,NaN,NaN,1063,2433,...,0,-4,NaN,2,NaN,fi211118b-06.pl2,12800.0,1,NaN,1
